# 📊 Évaluation XAI Étendue - Visualisations pour Thèse

## Objectifs
1. **Sélection d'images multi-critères** (faciles, difficiles, par classe)
2. **Évaluation statistique** sur plusieurs images
3. **Visualisations publication-ready** pour la thèse
4. **Analyse comparative CNN vs ViT**

---

## 1. Configuration et Imports

In [1]:
# ============================================
# IMPORTS
# ============================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import cv2
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import timm

from captum.attr import IntegratedGradients
from scipy import stats

# Configuration visualisation
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 150

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cpu


In [2]:
# ============================================
# CONFIGURATION
# ============================================

# Chemins - ADAPTER SELON VOTRE CONFIGURATION
DATA_DIR = '../data/ISIC2019'
RESULTS_DIR = '../results'
FIGURES_DIR = '../figures_these'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Classes ISIC 2019
CLASS_NAMES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
CLASS_FULL_NAMES = {
    'MEL': 'Mélanome',
    'NV': 'Naevus mélanocytaire',
    'BCC': 'Carcinome basocellulaire',
    'AK': 'Kératose actinique',
    'BKL': 'Kératose bénigne',
    'DF': 'Dermatofibrome',
    'VASC': 'Lésion vasculaire',
    'SCC': 'Carcinome épidermoïde'
}
NUM_CLASSES = len(CLASS_NAMES)

# Normalisation ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Configuration des modèles
MODELS_CONFIG = {
    'resnet50':        {'timm_name': 'resnet50',                     'type': 'CNN',         'batch_size': 32,  'lr': 1e-4,  'weights': '../results/resnet50_best.pth',              'color': '#1f77b4'},
    'efficientnet_b4': {'timm_name': 'efficientnet_b4',              'type': 'CNN',         'batch_size': 24,  'lr': 1e-4,  'weights': '../results/efficientnet_b4_best.pth',       'color': '#2ca02c'},
    'convnext_base':   {'timm_name': 'convnext_base',                'type': 'CNN-Modern',  'batch_size': 16,  'lr': 5e-5,  'weights': '../results/convnext_base_best.pth',         'color': '#9467bd'},
    'vit_base':        {'timm_name': 'vit_base_patch16_224',         'type': 'Transformer', 'batch_size': 16,  'lr': 1e-5,  'weights': '../results/vit_base_patch16_best.pth',      'color': '#d62728'},
    'deit_base':       {'timm_name': 'deit_base_patch16_224',        'type': 'Transformer', 'batch_size': 16,  'lr': 1e-5,  'weights': '../results/deit_base_patch16_best.pth',     'color': '#ff7f0e'},
    'swin_base':       {'timm_name': 'swin_base_patch4_window7_224', 'type': 'Transformer', 'batch_size': 16,  'lr': 1e-5,  'weights': '../results/swin_base_batchformer_best.pth', 'color': '#e377c2'},
}

print(f"Configuration chargée!")
print(f"   - Data dir: {DATA_DIR}")
print(f"   - Results dir: {RESULTS_DIR}")
print(f"   - Figures dir: {FIGURES_DIR}")

Configuration chargée!
   - Data dir: ../data/ISIC2019
   - Results dir: ../results
   - Figures dir: ../figures_these


## 2. Chargement des Modèles et Méthodes XAI

*(Copiez ici les classes ModelWrapper, GradCAMExplainer, etc. de votre notebook principal)*

In [3]:
# ============================================
# MODEL WRAPPER (copié de votre notebook)
# ============================================

class ModelWrapper(nn.Module):
    """Wrapper unifié pour CNN et ViT."""
    
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.model_name = model_name
        self.num_classes = num_classes
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
        self.model_type = self._get_model_type()
        
    def _get_model_type(self):
        vit_keywords = ['vit', 'deit', 'swin', 'beit']
        if any(kw in self.model_name.lower() for kw in vit_keywords):
            return 'transformer'
        return 'cnn'
    
    def forward(self, x):
        return self.model(x)
    
    def get_target_layer(self):
        if 'resnet' in self.model_name:
            return self.model.layer4[-1]
        elif 'efficientnet' in self.model_name:
            return self.model.conv_head
        elif 'vit' in self.model_name or 'deit' in self.model_name:
            return self.model.blocks[-1].norm1
        elif 'swin' in self.model_name:
            return self.model.layers[-1].blocks[-1].norm1
        elif 'convnext' in self.model_name:
            return self.model.stages[-1].blocks[-1]
        else:
            return list(self.model.children())[-2]

print("ModelWrapper défini")

ModelWrapper défini


In [4]:
# ============================================
# GRAD-CAM EXPLAINER
# ============================================

class GradCAMExplainer:
    def __init__(self, model, device='cuda'):
        self.model = model
        self.device = device
        self.target_layer = model.get_target_layer()
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def explain(self, input_tensor, target_class=None):
        self.model.eval()
        input_tensor = input_tensor.to(self.device)
        input_tensor.requires_grad = True
        
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)
        
        gradients = self.gradients
        activations = self.activations
        
        if self.model.model_type == 'transformer':
            if len(activations.shape) == 3:
                num_patches = activations.shape[1]
                if num_patches == 197:
                    activations = activations[:, 1:, :]
                    gradients = gradients[:, 1:, :]
                    num_patches = 196
                h = w = int(np.sqrt(num_patches))
                activations = activations.reshape(1, h, w, -1).permute(0, 3, 1, 2)
                gradients = gradients.reshape(1, h, w, -1).permute(0, 3, 1, 2)
        
        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

print("GradCAMExplainer défini")

GradCAMExplainer défini


In [5]:
# ============================================
# INTEGRATED GRADIENTS EXPLAINER
# ============================================

class IntegratedGradientsExplainer:
    def __init__(self, model, device='cuda', n_steps=50):
        self.model = model
        self.device = device
        self.n_steps = n_steps
        self.ig = IntegratedGradients(model)
    
    def explain(self, input_tensor, target_class=None, baseline=None):
        self.model.eval()
        input_tensor = input_tensor.to(self.device)
        
        if target_class is None:
            with torch.no_grad():
                output = self.model(input_tensor)
                target_class = output.argmax(dim=1).item()
        
        if baseline is None:
            baseline = torch.zeros_like(input_tensor).to(self.device)
        
        attributions = self.ig.attribute(
            input_tensor, baselines=baseline, target=target_class,
            n_steps=self.n_steps, return_convergence_delta=False
        )
        
        attr = attributions.squeeze().cpu().detach().numpy()
        saliency = np.abs(attr).sum(axis=0)
        saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
        return saliency

print("IntegratedGradientsExplainer défini")

IntegratedGradientsExplainer défini


In [6]:
# ============================================
# ATTENTION ROLLOUT EXPLAINER
# ============================================

class AttentionRolloutExplainer:
    def __init__(self, model, device='cuda'):
        if model.model_type != 'transformer':
            raise ValueError("Attention Rollout uniquement pour les Transformers")
        self.model = model
        self.device = device

    def explain(self, input_tensor, target_class=None, head_fusion='mean'):
        self.model.eval()
        input_tensor = input_tensor.to(self.device)
        attentions = []

        def hook_fn(module, input, output):
            B, N, C = input[0].shape
            qkv = module.qkv(input[0]).reshape(B, N, 3, module.num_heads, C // module.num_heads)
            qkv = qkv.permute(2, 0, 3, 1, 4)
            q, k, v = qkv[0], qkv[1], qkv[2]
            attn = (q @ k.transpose(-2, -1)) * (C // module.num_heads) ** -0.5
            attn = attn.softmax(dim=-1)
            attentions.append(attn.detach().cpu())

        hooks = []
        for block in self.model.model.blocks:
            hooks.append(block.attn.register_forward_hook(hook_fn))

        with torch.no_grad():
            _ = self.model(input_tensor)

        for hook in hooks:
            hook.remove()

        result = torch.eye(attentions[0].shape[-1])
        for attention in attentions:
            if head_fusion == 'mean':
                attention_heads_fused = attention.mean(dim=1)
            elif head_fusion == 'max':
                attention_heads_fused = attention.max(dim=1)[0]
            else:
                attention_heads_fused = attention.min(dim=1)[0]
            
            I = torch.eye(attention_heads_fused.shape[-1])
            a = (attention_heads_fused + I) / 2
            a = a / a.sum(dim=-1, keepdim=True)
            result = torch.matmul(a.squeeze(0), result)

        mask = result[0, 1:]
        num_patches = mask.shape[0]
        h = w = int(np.sqrt(num_patches))
        mask = mask.reshape(h, w).numpy()
        mask = cv2.resize(mask, (224, 224))
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        return mask

print("AttentionRolloutExplainer défini")

AttentionRolloutExplainer défini


In [7]:
# ============================================
# INSERTION / DELETION AUC
# ============================================

def compute_insertion_deletion(model, input_tensor, saliency_map, target_class, num_steps=100, device='cuda'):
    """Calculer les scores d'Insertion et Deletion AUC."""
    model.eval()
    input_tensor = input_tensor.to(device)

    if saliency_map.shape != (224, 224):
        saliency_map = cv2.resize(saliency_map, (224, 224))

    flat_saliency = saliency_map.flatten()
    sorted_indices = np.argsort(flat_saliency)[::-1]

    img_np = input_tensor.squeeze().cpu().detach().numpy()
    baseline = np.zeros_like(img_np)

    num_pixels = 224 * 224
    step_size = num_pixels // num_steps

    insertion_scores = []
    deletion_scores = []

    for step in range(num_steps + 1):
        num_reveal = min(step * step_size, num_pixels)
        mask = np.zeros(num_pixels, dtype=bool)
        mask[sorted_indices[:num_reveal]] = True
        mask = mask.reshape(224, 224)

        insertion_img = baseline.copy()
        deletion_img = img_np.copy()
        for c in range(3):
            insertion_img[c][mask] = img_np[c][mask]
            deletion_img[c][mask] = baseline[c][mask]

        with torch.no_grad():
            ins_tensor = torch.from_numpy(insertion_img).unsqueeze(0).float().to(device)
            del_tensor = torch.from_numpy(deletion_img).unsqueeze(0).float().to(device)
            ins_output = F.softmax(model(ins_tensor), dim=1)
            del_output = F.softmax(model(del_tensor), dim=1)
            insertion_scores.append(ins_output[0, target_class].item())
            deletion_scores.append(del_output[0, target_class].item())

    x = np.linspace(0, 1, len(insertion_scores))
    insertion_auc = np.trapz(insertion_scores, x)
    deletion_auc = np.trapz(deletion_scores, x)

    return insertion_auc, deletion_auc, insertion_scores, deletion_scores

print("Fonction compute_insertion_deletion définie")

Fonction compute_insertion_deletion définie


## 3. Sélection Multi-Critères des Images

In [8]:
# ============================================
# SÉLECTION D'IMAGES MULTI-CRITÈRES
# ============================================

def load_and_preprocess_image(image_path):
    """Charger et prétraiter une image."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
    
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0)
    
    # Image originale pour visualisation
    img_display = np.array(img.resize((224, 224))) / 255.0
    
    return img_tensor, img_display


def get_model_predictions(models, image_tensor, device):
    """Obtenir les prédictions de tous les modèles."""
    predictions = {}
    
    for model_name, model in models.items():
        model.eval()
        with torch.no_grad():
            output = model(image_tensor.to(device))
            probs = F.softmax(output, dim=1)
            pred_class = probs.argmax(dim=1).item()
            confidence = probs.max().item()
            
            predictions[model_name] = {
                'class': pred_class,
                'class_name': CLASS_NAMES[pred_class],
                'confidence': confidence,
                'all_probs': probs.cpu().numpy()[0]
            }
    
    return predictions


def select_evaluation_images(df, img_dir, models, device, n_per_category=3):
    """
    Sélectionner des images selon différents critères.
    
    Catégories:
    1. Consensus élevé: Tous les modèles corrects avec confiance > 80%
    2. Consensus moyen: Majorité correcte, confiance 50-80%
    3. Désaccord: Les modèles ne sont pas d'accord
    4. Cas difficiles: Tous les modèles se trompent ou confiance < 50%
    5. Par classe: 1 image par classe ISIC
    """
    
    selected_images = {
        'high_consensus': [],
        'medium_consensus': [],
        'disagreement': [],
        'difficult': [],
        'per_class': {cls: [] for cls in CLASS_NAMES}
    }
    
    print("Analyse des images pour sélection...")
    
    # Échantillonner des images
    sample_size = min(500, len(df))
    sampled_df = df.sample(n=sample_size, random_state=SEED)
    
    for idx, row in tqdm(sampled_df.iterrows(), total=len(sampled_df)):
        image_id = row['image']
        true_class = row['label']
        true_class_name = CLASS_NAMES[true_class]
        
        image_path = os.path.join(img_dir, f"{image_id}.jpg")
        if not os.path.exists(image_path):
            continue
        
        try:
            img_tensor, img_display = load_and_preprocess_image(image_path)
            predictions = get_model_predictions(models, img_tensor, device)
            
            # Analyser les prédictions
            correct_count = sum(1 for p in predictions.values() if p['class'] == true_class)
            avg_confidence = np.mean([p['confidence'] for p in predictions.values()])
            pred_classes = [p['class'] for p in predictions.values()]
            agreement = len(set(pred_classes)) == 1
            
            image_info = {
                'image_id': image_id,
                'image_path': image_path,
                'true_class': true_class,
                'true_class_name': true_class_name,
                'correct_count': correct_count,
                'avg_confidence': avg_confidence,
                'agreement': agreement,
                'predictions': predictions
            }
            
            # Catégoriser
            if correct_count == len(models) and avg_confidence > 0.8:
                if len(selected_images['high_consensus']) < n_per_category:
                    selected_images['high_consensus'].append(image_info)
            
            elif correct_count >= len(models) // 2 and 0.5 <= avg_confidence <= 0.8:
                if len(selected_images['medium_consensus']) < n_per_category:
                    selected_images['medium_consensus'].append(image_info)
            
            elif not agreement:
                if len(selected_images['disagreement']) < n_per_category:
                    selected_images['disagreement'].append(image_info)
            
            elif correct_count == 0 or avg_confidence < 0.5:
                if len(selected_images['difficult']) < n_per_category:
                    selected_images['difficult'].append(image_info)
            
            # Par classe
            if len(selected_images['per_class'][true_class_name]) < 2:
                if correct_count >= len(models) // 2:
                    selected_images['per_class'][true_class_name].append(image_info)
                    
        except Exception as e:
            continue
    
    # Résumé
    print("\n" + "="*50)
    print("IMAGES SÉLECTIONNÉES:")
    print("="*50)
    for category, images in selected_images.items():
        if category == 'per_class':
            total = sum(len(v) for v in images.values())
            print(f"   {category}: {total} images")
        else:
            print(f"   {category}: {len(images)} images")
    
    return selected_images

print("Fonctions de sélection d'images définies")

Fonctions de sélection d'images définies


## 4. Chargement des Modèles et Sélection des Images

In [11]:
# ============================================
# CHARGER LES 6 MODÈLES (avec poids entraînés)
# ============================================

def remap_state_dict(state_dict, model_name):
    """
    Adapte le state_dict selon l'architecture source du checkpoint.
    
    - swin_base : sauvegardé depuis SwinWithBatchFormerWrapper
        → clés "backbone.XXX" + "batch_former.*" + "head.weight/bias"
        → on strip "backbone.", remplace "head.weight/bias" → "head.fc.weight/bias",
          ignore "batch_former.*"
    - autres : sauvegardé depuis ModelWrapper.model (timm)
        → clés directes, rien à faire
    """
    if model_name != 'swin_base':
        return state_dict

    new_sd = {}
    for k, v in state_dict.items():
        if k.startswith('batch_former.'):
            continue  # module spécifique BatchFormer, inutile pour l'inférence
        elif k.startswith('backbone.'):
            new_sd[k[len('backbone.'):]] = v
        elif k == 'head.weight':
            new_sd['head.fc.weight'] = v
        elif k == 'head.bias':
            new_sd['head.fc.bias'] = v
        else:
            new_sd[k] = v
    return new_sd


print("Chargement des 6 modèles...\n")

NOTEBOOK_DIR = os.getcwd()

models = {}
for model_name, config in MODELS_CONFIG.items():
    print(f"Loading {model_name}...")
    try:
        model = ModelWrapper(config['timm_name'], NUM_CLASSES, pretrained=False).to(device)

        weights_path = os.path.normpath(os.path.join(NOTEBOOK_DIR, config['weights']))

        if os.path.exists(weights_path):
            ckpt = torch.load(weights_path, map_location=device, weights_only=False)
            raw_sd = ckpt.get('model_state_dict', ckpt)
            state_dict = remap_state_dict(raw_sd, model_name)
            model.model.load_state_dict(state_dict)

            val_acc = ckpt.get('val_acc', '?')
            epoch   = ckpt.get('epoch', '?')
            val_acc_str = f"{val_acc:.4f}" if isinstance(val_acc, float) else str(val_acc)
            print(f"   ✓ Poids chargés — epoch={epoch}, val_acc={val_acc_str}")
        else:
            print(f"   ⚠ Poids introuvables: {weights_path}")

        model.eval()
        models[model_name] = model
        params = sum(p.numel() for p in model.parameters())
        print(f"   ✓ {params/1e6:.1f}M paramètres")

    except Exception as e:
        print(f"   ✗ Erreur: {e}")

print(f"\n✓ {len(models)} modèles chargés!")

Chargement des 6 modèles...

Loading resnet50...
   ✓ Poids chargés — epoch=19, val_acc=0.8282
   ✓ 23.5M paramètres
Loading efficientnet_b4...
   ✓ Poids chargés — epoch=7, val_acc=0.7873
   ✓ 17.6M paramètres
Loading convnext_base...
   ✓ Poids chargés — epoch=21, val_acc=0.8582
   ✓ 87.6M paramètres
Loading vit_base...
   ✓ Poids chargés — epoch=28, val_acc=0.7890
   ✓ 85.8M paramètres
Loading deit_base...
   ✓ Poids chargés — epoch=27, val_acc=0.8323
   ✓ 85.8M paramètres
Loading swin_base...
   ✓ Poids chargés — epoch=1, val_acc=0.8471
   ✓ 86.8M paramètres

✓ 6 modèles chargés!


In [12]:
# ============================================
# CHARGER LE DATASET ET SÉLECTIONNER LES IMAGES
# ============================================

# Charger les labels
csv_path = os.path.join(DATA_DIR, 'ISIC_2019_Training_GroundTruth.csv')
df = pd.read_csv(csv_path)

# Convertir one-hot en labels
label_columns = [col for col in df.columns if col != 'image' and col != 'UNK']
df['label'] = df[label_columns].values.argmax(axis=1)

print(f"Dataset: {len(df)} images")

# Sélectionner les images
img_dir = os.path.join(DATA_DIR, 'ISIC_2019_Training_Input')
selected_images = select_evaluation_images(df, img_dir, models, device, n_per_category=5)

Dataset: 25331 images
Analyse des images pour sélection...


  0%|          | 0/500 [00:00<?, ?it/s]


IMAGES SÉLECTIONNÉES:
   high_consensus: 5 images
   medium_consensus: 5 images
   disagreement: 5 images
   difficult: 1 images
   per_class: 16 images


## 5. Évaluation XAI sur Plusieurs Images

In [13]:
# ============================================
# ÉVALUATION XAI COMPLÈTE
# ============================================

def evaluate_xai_on_image(image_info, models, device):
    """Évaluer toutes les méthodes XAI sur une image."""
    
    img_tensor, img_display = load_and_preprocess_image(image_info['image_path'])
    img_tensor = img_tensor.to(device)
    
    results = []
    
    for model_name, model in models.items():
        pred_class = image_info['predictions'][model_name]['class']
        confidence = image_info['predictions'][model_name]['confidence']
        is_correct = pred_class == image_info['true_class']
        
        # Grad-CAM
        try:
            gradcam = GradCAMExplainer(model, device)
            gradcam_map = gradcam.explain(img_tensor, pred_class)
            ins_auc, del_auc, _, _ = compute_insertion_deletion(
                model, img_tensor, gradcam_map, pred_class, device=device
            )
            results.append({
                'image_id': image_info['image_id'],
                'model': model_name,
                'model_type': MODELS_CONFIG[model_name]['type'],
                'method': 'GradCAM',
                'insertion_auc': ins_auc,
                'deletion_auc': del_auc,
                'faithfulness': ins_auc - del_auc,
                'confidence': confidence,
                'is_correct': is_correct,
                'true_class': image_info['true_class_name']
            })
        except:
            pass
        
        # Integrated Gradients
        try:
            ig = IntegratedGradientsExplainer(model, device, n_steps=30)
            ig_map = ig.explain(img_tensor, pred_class)
            ins_auc, del_auc, _, _ = compute_insertion_deletion(
                model, img_tensor, ig_map, pred_class, device=device
            )
            results.append({
                'image_id': image_info['image_id'],
                'model': model_name,
                'model_type': MODELS_CONFIG[model_name]['type'],
                'method': 'IntegratedGradients',
                'insertion_auc': ins_auc,
                'deletion_auc': del_auc,
                'faithfulness': ins_auc - del_auc,
                'confidence': confidence,
                'is_correct': is_correct,
                'true_class': image_info['true_class_name']
            })
        except:
            pass
        
        # Attention Rollout (ViT/DeiT uniquement)
        if model_name in ['vit_base', 'deit_base']:
            try:
                attn = AttentionRolloutExplainer(model, device)
                attn_map = attn.explain(img_tensor)
                ins_auc, del_auc, _, _ = compute_insertion_deletion(
                    model, img_tensor, attn_map, pred_class, device=device
                )
                results.append({
                    'image_id': image_info['image_id'],
                    'model': model_name,
                    'model_type': MODELS_CONFIG[model_name]['type'],
                    'method': 'AttentionRollout',
                    'insertion_auc': ins_auc,
                    'deletion_auc': del_auc,
                    'faithfulness': ins_auc - del_auc,
                    'confidence': confidence,
                    'is_correct': is_correct,
                    'true_class': image_info['true_class_name']
                })
            except:
                pass
    
    return results


def run_full_evaluation(selected_images, models, device):
    """Exécuter l'évaluation complète sur toutes les images sélectionnées."""
    
    all_results = []
    
    # Collecter toutes les images
    all_images = []
    for category, images in selected_images.items():
        if category == 'per_class':
            for cls_images in images.values():
                for img_info in cls_images:
                    img_info['category'] = 'per_class'
                    all_images.append(img_info)
        else:
            for img_info in images:
                img_info['category'] = category
                all_images.append(img_info)
    
    print(f"\nÉvaluation XAI sur {len(all_images)} images...\n")
    
    for img_info in tqdm(all_images):
        try:
            results = evaluate_xai_on_image(img_info, models, device)
            for r in results:
                r['category'] = img_info['category']
            all_results.extend(results)
        except Exception as e:
            print(f"Erreur sur {img_info['image_id']}: {e}")
    
    return pd.DataFrame(all_results)

print("Fonctions d'évaluation définies")

Fonctions d'évaluation définies


In [ ]:
# ============================================
# LANCER L'ÉVALUATION COMPLÈTE
# ============================================

results_df = run_full_evaluation(selected_images, models, device)

# Sauvegarder les résultats
results_df.to_csv(os.path.join(RESULTS_DIR, 'xai_evaluation_extended.csv'), index=False)

print(f"\n✓ Évaluation terminée: {len(results_df)} résultats")
print(f"✓ Résultats sauvegardés: {RESULTS_DIR}/xai_evaluation_extended.csv")


Évaluation XAI sur 32 images...



  0%|          | 0/32 [00:00<?, ?it/s]

## 6. Visualisations pour la Thèse

In [ ]:
# ============================================
# FIGURE 1: COMPARAISON CNN vs TRANSFORMERS
# ============================================

def plot_cnn_vs_transformer(results_df, save_path=None):
    """Comparer la fidélité entre CNN et Transformers."""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Couleurs par type
    type_colors = {'CNN': '#1f77b4', 'CNN-Modern': '#9467bd', 'Transformer': '#d62728'}
    
    # 1. Faithfulness par type de modèle
    ax1 = axes[0]
    type_faith = results_df.groupby('model_type')['faithfulness'].agg(['mean', 'std']).reset_index()
    colors = [type_colors[t] for t in type_faith['model_type']]
    bars = ax1.bar(type_faith['model_type'], type_faith['mean'], yerr=type_faith['std'],
                   color=colors, capsize=5, edgecolor='black', linewidth=1.5)
    ax1.set_ylabel('Faithfulness Score', fontsize=12)
    ax1.set_title('Fidélité Moyenne par Type d\'Architecture', fontweight='bold', fontsize=14)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_ylim(-0.1, 0.6)
    
    # 2. Faithfulness par modèle
    ax2 = axes[1]
    model_faith = results_df.groupby(['model', 'model_type'])['faithfulness'].mean().reset_index()
    model_faith = model_faith.sort_values('faithfulness', ascending=True)
    colors = [type_colors[t] for t in model_faith['model_type']]
    bars = ax2.barh(model_faith['model'], model_faith['faithfulness'], color=colors, edgecolor='black')
    ax2.set_xlabel('Faithfulness Score', fontsize=12)
    ax2.set_title('Fidélité par Modèle', fontweight='bold', fontsize=14)
    ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    
    # Légende
    patches = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
    ax2.legend(handles=patches, loc='lower right')
    
    # 3. Distribution Faithfulness
    ax3 = axes[2]
    for model_type in ['CNN', 'CNN-Modern', 'Transformer']:
        data = results_df[results_df['model_type'] == model_type]['faithfulness']
        ax3.hist(data, bins=20, alpha=0.6, label=model_type, color=type_colors[model_type])
    ax3.set_xlabel('Faithfulness Score', fontsize=12)
    ax3.set_ylabel('Fréquence', fontsize=12)
    ax3.set_title('Distribution de la Fidélité', fontweight='bold', fontsize=14)
    ax3.legend()
    ax3.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Figure sauvegardée: {save_path}")
    
    plt.show()

# Générer la figure
plot_cnn_vs_transformer(results_df, os.path.join(FIGURES_DIR, 'fig1_cnn_vs_transformer.png'))

In [ ]:
# ============================================
# FIGURE 2: COMPARAISON DES MÉTHODES XAI
# ============================================

def plot_xai_methods_comparison(results_df, save_path=None):
    """Comparer les méthodes XAI."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    method_colors = {
        'GradCAM': '#2ca02c',
        'IntegratedGradients': '#ff7f0e',
        'AttentionRollout': '#d62728'
    }
    
    # 1. Heatmap: Modèle x Méthode
    ax1 = axes[0, 0]
    pivot = results_df.pivot_table(
        index='model', columns='method', values='faithfulness', aggfunc='mean'
    ).fillna(0)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax1,
                cbar_kws={'label': 'Faithfulness'}, linewidths=0.5)
    ax1.set_title('Faithfulness par Modèle et Méthode XAI', fontweight='bold', fontsize=14)
    
    # 2. Insertion vs Deletion AUC
    ax2 = axes[0, 1]
    for method in results_df['method'].unique():
        data = results_df[results_df['method'] == method]
        ax2.scatter(data['insertion_auc'], data['deletion_auc'], 
                   label=method, alpha=0.6, s=50, color=method_colors.get(method, 'gray'))
    ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='y=x')
    ax2.set_xlabel('Insertion AUC (↑ meilleur)', fontsize=12)
    ax2.set_ylabel('Deletion AUC (↓ meilleur)', fontsize=12)
    ax2.set_title('Insertion vs Deletion AUC', fontweight='bold', fontsize=14)
    ax2.legend()
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    # 3. Boxplot par méthode
    ax3 = axes[1, 0]
    methods_order = ['GradCAM', 'IntegratedGradients', 'AttentionRollout']
    colors = [method_colors.get(m, 'gray') for m in methods_order if m in results_df['method'].unique()]
    sns.boxplot(data=results_df, x='method', y='faithfulness', ax=ax3, palette=colors,
               order=[m for m in methods_order if m in results_df['method'].unique()])
    ax3.set_xlabel('Méthode XAI', fontsize=12)
    ax3.set_ylabel('Faithfulness Score', fontsize=12)
    ax3.set_title('Distribution de la Fidélité par Méthode', fontweight='bold', fontsize=14)
    ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # 4. Meilleure méthode par modèle
    ax4 = axes[1, 1]
    best_method = results_df.groupby(['model', 'method'])['faithfulness'].mean().reset_index()
    best_method = best_method.loc[best_method.groupby('model')['faithfulness'].idxmax()]
    colors = [method_colors.get(m, 'gray') for m in best_method['method']]
    bars = ax4.barh(best_method['model'], best_method['faithfulness'], color=colors, edgecolor='black')
    
    # Annoter avec le nom de la méthode
    for i, (idx, row) in enumerate(best_method.iterrows()):
        ax4.text(row['faithfulness'] + 0.01, i, row['method'], va='center', fontsize=10)
    
    ax4.set_xlabel('Faithfulness Score', fontsize=12)
    ax4.set_title('Meilleure Méthode XAI par Modèle', fontweight='bold', fontsize=14)
    ax4.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Figure sauvegardée: {save_path}")
    
    plt.show()

# Générer la figure
plot_xai_methods_comparison(results_df, os.path.join(FIGURES_DIR, 'fig2_xai_methods_comparison.png'))

In [ ]:
# ============================================
# FIGURE 3: ANALYSE PAR CATÉGORIE D'IMAGE
# ============================================

def plot_category_analysis(results_df, save_path=None):
    """Analyser la fidélité selon la catégorie d'image."""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    category_colors = {
        'high_consensus': '#2ca02c',
        'medium_consensus': '#ff7f0e',
        'disagreement': '#d62728',
        'difficult': '#7f7f7f',
        'per_class': '#9467bd'
    }
    
    category_labels = {
        'high_consensus': 'Consensus élevé\n(Tous corrects >80%)',
        'medium_consensus': 'Consensus moyen\n(Majorité correcte 50-80%)',
        'disagreement': 'Désaccord\n(Modèles divergent)',
        'difficult': 'Cas difficiles\n(Erreurs ou faible confiance)',
        'per_class': 'Par classe'
    }
    
    # 1. Faithfulness par catégorie
    ax1 = axes[0]
    cat_faith = results_df.groupby('category')['faithfulness'].agg(['mean', 'std']).reset_index()
    cat_faith['label'] = cat_faith['category'].map(category_labels)
    colors = [category_colors.get(c, 'gray') for c in cat_faith['category']]
    bars = ax1.bar(range(len(cat_faith)), cat_faith['mean'], yerr=cat_faith['std'],
                   color=colors, capsize=5, edgecolor='black')
    ax1.set_xticks(range(len(cat_faith)))
    ax1.set_xticklabels(cat_faith['label'], rotation=45, ha='right', fontsize=10)
    ax1.set_ylabel('Faithfulness Score', fontsize=12)
    ax1.set_title('Fidélité selon le Type d\'Image', fontweight='bold', fontsize=14)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # 2. Corrélation Confiance vs Faithfulness
    ax2 = axes[1]
    for model_type in ['CNN', 'CNN-Modern', 'Transformer']:
        data = results_df[results_df['model_type'] == model_type]
        ax2.scatter(data['confidence'], data['faithfulness'], alpha=0.5, label=model_type, s=30)
    
    # Ligne de régression
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        results_df['confidence'], results_df['faithfulness']
    )
    x_line = np.linspace(0, 1, 100)
    ax2.plot(x_line, slope * x_line + intercept, 'k--', 
             label=f'Régression (R²={r_value**2:.3f})')
    
    ax2.set_xlabel('Confiance du Modèle', fontsize=12)
    ax2.set_ylabel('Faithfulness Score', fontsize=12)
    ax2.set_title('Relation Confiance vs Fidélité', fontweight='bold', fontsize=14)
    ax2.legend()
    ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
    
    # 3. Prédiction correcte vs incorrecte
    ax3 = axes[2]
    correct_faith = results_df.groupby(['model_type', 'is_correct'])['faithfulness'].mean().reset_index()
    
    x = np.arange(len(correct_faith['model_type'].unique()))
    width = 0.35
    
    correct_data = correct_faith[correct_faith['is_correct'] == True]
    incorrect_data = correct_faith[correct_faith['is_correct'] == False]
    
    ax3.bar(x - width/2, correct_data['faithfulness'], width, label='Prédiction correcte', color='#2ca02c')
    ax3.bar(x + width/2, incorrect_data['faithfulness'], width, label='Prédiction incorrecte', color='#d62728')
    
    ax3.set_xticks(x)
    ax3.set_xticklabels(correct_data['model_type'])
    ax3.set_ylabel('Faithfulness Score', fontsize=12)
    ax3.set_title('Fidélité: Correct vs Incorrect', fontweight='bold', fontsize=14)
    ax3.legend()
    ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Figure sauvegardée: {save_path}")
    
    plt.show()

# Générer la figure
plot_category_analysis(results_df, os.path.join(FIGURES_DIR, 'fig3_category_analysis.png'))

In [ ]:
# ============================================
# FIGURE 4: TABLEAU STATISTIQUE COMPLET
# ============================================

def create_summary_table(results_df, save_path=None):
    """Créer un tableau récapitulatif statistique."""
    
    # Statistiques par modèle et méthode
    summary = results_df.groupby(['model', 'model_type', 'method']).agg({
        'insertion_auc': ['mean', 'std'],
        'deletion_auc': ['mean', 'std'],
        'faithfulness': ['mean', 'std', 'min', 'max']
    }).round(4)
    
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary = summary.reset_index()
    
    # Afficher
    print("\n" + "="*100)
    print("                    TABLEAU RÉCAPITULATIF STATISTIQUE")
    print("="*100)
    
    display_cols = ['model', 'model_type', 'method', 
                   'faithfulness_mean', 'faithfulness_std',
                   'insertion_auc_mean', 'deletion_auc_mean']
    
    print(summary[display_cols].to_string(index=False))
    print("="*100)
    
    # Sauvegarder
    if save_path:
        summary.to_csv(save_path, index=False)
        print(f"\n✓ Tableau sauvegardé: {save_path}")
    
    return summary

# Créer le tableau
summary_table = create_summary_table(results_df, os.path.join(RESULTS_DIR, 'xai_summary_statistics.csv'))

In [ ]:
# ============================================
# FIGURE 5: VISUALISATION XAI SUR IMAGE EXEMPLE
# ============================================

def visualize_xai_example(image_info, models, device, save_path=None):
    """Visualiser les cartes XAI pour une image."""
    
    img_tensor, img_display = load_and_preprocess_image(image_info['image_path'])
    img_tensor = img_tensor.to(device)
    
    n_models = len(models)
    fig, axes = plt.subplots(3, n_models + 1, figsize=(4*(n_models+1), 12))
    
    # Ligne 0: Image originale + prédictions
    axes[0, 0].imshow(img_display)
    axes[0, 0].set_title(f"Original\nClasse vraie: {image_info['true_class_name']}", fontsize=11)
    axes[0, 0].axis('off')
    
    for i, (model_name, model) in enumerate(models.items()):
        pred = image_info['predictions'][model_name]
        is_correct = pred['class'] == image_info['true_class']
        color = 'green' if is_correct else 'red'
        
        axes[0, i+1].imshow(img_display)
        axes[0, i+1].set_title(f"{model_name}\n{pred['class_name']} ({pred['confidence']:.1%})",
                               color=color, fontsize=10)
        axes[0, i+1].axis('off')
    
    # Ligne 1: Grad-CAM
    axes[1, 0].text(0.5, 0.5, 'Grad-CAM', ha='center', va='center', fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    for i, (model_name, model) in enumerate(models.items()):
        try:
            gradcam = GradCAMExplainer(model, device)
            pred_class = image_info['predictions'][model_name]['class']
            cam = gradcam.explain(img_tensor, pred_class)
            
            heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
            heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
            overlay = 0.5 * heatmap + 0.5 * img_display
            
            axes[1, i+1].imshow(np.clip(overlay, 0, 1))
        except:
            axes[1, i+1].imshow(img_display, alpha=0.3)
        axes[1, i+1].axis('off')
    
    # Ligne 2: Integrated Gradients
    axes[2, 0].text(0.5, 0.5, 'Integrated\nGradients', ha='center', va='center', fontsize=14, fontweight='bold')
    axes[2, 0].axis('off')
    
    for i, (model_name, model) in enumerate(models.items()):
        try:
            ig = IntegratedGradientsExplainer(model, device, n_steps=30)
            pred_class = image_info['predictions'][model_name]['class']
            saliency = ig.explain(img_tensor, pred_class)
            
            heatmap = cv2.applyColorMap(np.uint8(255 * saliency), cv2.COLORMAP_JET)
            heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
            overlay = 0.5 * heatmap + 0.5 * img_display
            
            axes[2, i+1].imshow(np.clip(overlay, 0, 1))
        except:
            axes[2, i+1].imshow(img_display, alpha=0.3)
        axes[2, i+1].axis('off')
    
    plt.suptitle(f"Comparaison XAI - Image: {image_info['image_id']}", fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Figure sauvegardée: {save_path}")
    
    plt.show()

# Visualiser un exemple de chaque catégorie
for category in ['high_consensus', 'disagreement', 'difficult']:
    if selected_images.get(category) and len(selected_images[category]) > 0:
        img_info = selected_images[category][0]
        print(f"\n{'='*60}")
        print(f"Catégorie: {category}")
        print(f"{'='*60}")
        visualize_xai_example(
            img_info, models, device,
            os.path.join(FIGURES_DIR, f'fig5_xai_example_{category}.png')
        )

## 7. Conclusions et Recommandations

In [ ]:
# ============================================
# CONCLUSIONS AUTOMATIQUES
# ============================================

def generate_conclusions(results_df):
    """Générer des conclusions basées sur les résultats."""
    
    print("\n" + "="*80)
    print("                    CONCLUSIONS DE L'ÉVALUATION XAI")
    print("="*80)
    
    # 1. Meilleur type d'architecture
    type_ranking = results_df.groupby('model_type')['faithfulness'].mean().sort_values(ascending=False)
    print("\n📊 CLASSEMENT PAR TYPE D'ARCHITECTURE:")
    print("-"*50)
    for i, (arch_type, faith) in enumerate(type_ranking.items(), 1):
        emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉"
        print(f"   {emoji} {i}. {arch_type:15s}: Faithfulness = {faith:.4f}")
    
    # 2. Meilleur modèle global
    model_ranking = results_df.groupby('model')['faithfulness'].mean().sort_values(ascending=False)
    print("\n📊 CLASSEMENT DES MODÈLES:")
    print("-"*50)
    for i, (model, faith) in enumerate(model_ranking.items(), 1):
        model_type = MODELS_CONFIG[model]['type']
        emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
        print(f"   {emoji} {i}. {model:20s} ({model_type:12s}): {faith:.4f}")
    
    # 3. Meilleure méthode XAI
    method_ranking = results_df.groupby('method')['faithfulness'].mean().sort_values(ascending=False)
    print("\n📊 CLASSEMENT DES MÉTHODES XAI:")
    print("-"*50)
    for i, (method, faith) in enumerate(method_ranking.items(), 1):
        emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉"
        print(f"   {emoji} {i}. {method:25s}: {faith:.4f}")
    
    # 4. Meilleure combinaison modèle + méthode
    best_combo = results_df.groupby(['model', 'method'])['faithfulness'].mean().idxmax()
    best_faith = results_df.groupby(['model', 'method'])['faithfulness'].mean().max()
    print("\n🏆 MEILLEURE COMBINAISON:")
    print("-"*50)
    print(f"   {best_combo[0]} + {best_combo[1]}")
    print(f"   Faithfulness = {best_faith:.4f}")
    
    # 5. Recommandations
    print("\n💡 RECOMMANDATIONS POUR L'IMAGERIE MÉDICALE:")
    print("-"*50)
    
    best_type = type_ranking.index[0]
    best_model = model_ranking.index[0]
    best_method = method_ranking.index[0]
    
    print(f"   1. Architecture recommandée : {best_type}")
    print(f"   2. Modèle recommandé        : {best_model}")
    print(f"   3. Méthode XAI recommandée  : {best_method}")
    
    # Tests statistiques
    cnn_faith = results_df[results_df['model_type'].isin(['CNN', 'CNN-Modern'])]['faithfulness']
    vit_faith = results_df[results_df['model_type'] == 'Transformer']['faithfulness']
    t_stat, p_value = stats.ttest_ind(cnn_faith, vit_faith)
    
    print("\n📈 TEST STATISTIQUE (CNN vs Transformer):")
    print("-"*50)
    print(f"   t-statistic = {t_stat:.4f}")
    print(f"   p-value     = {p_value:.6f}")
    if p_value < 0.05:
        print("   → Différence SIGNIFICATIVE (p < 0.05)")
    else:
        print("   → Différence non significative (p >= 0.05)")
    
    print("\n" + "="*80)

# Générer les conclusions
generate_conclusions(results_df)

In [ ]:
# ============================================
# LISTE DES FIGURES GÉNÉRÉES
# ============================================

print("\n" + "="*60)
print("       FIGURES GÉNÉRÉES POUR LA THÈSE")
print("="*60)

figures_list = [
    ("fig1_cnn_vs_transformer.png", "Comparaison CNN vs Transformer"),
    ("fig2_xai_methods_comparison.png", "Comparaison des méthodes XAI"),
    ("fig3_category_analysis.png", "Analyse par catégorie d'image"),
    ("fig5_xai_example_*.png", "Exemples de visualisation XAI")
]

for filename, description in figures_list:
    print(f"   📊 {filename}")
    print(f"      → {description}")

print("\n" + "="*60)
print(f"Dossier: {FIGURES_DIR}")
print("="*60)